# 4. Integral Control

[Previous lesson](03_proportional_control.ipynb) · [Next lesson](05_derivative_control.ipynb)

A proportional controller needs persistent error to keep producing voltage. Integral action can remember past error and sustain voltage even when the present error has vanished.

Work through the explanations, then implement each exercise. Hints and complete reference solutions are collapsed below your code cells. Solutions build in order; Run All does not execute them.

**Prerequisites:** complete the preceding lessons. This notebook starts independently: run the setup cell once before attempting the exercises.

## Previously implemented code

This setup repeats earlier material so you do not need another notebook running. Read it as a recap, then run it. The new work for this lesson remains in the empty cells below.

**Setup inventory:** `motor_dynamics` evaluates the two motor derivatives using SI units and state order `[current, omega]`. Where supplied, `simulate` advances the sampled controller and motor separately, and `plot_run` displays speed, voltage, and term contributions. Controller classes in setup were taught previously; their histories start fresh when you construct an instance. Run setup after restarting the kernel, then execute your exercise cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

R, L, K_t, K_e, J, b = 1.0, 0.01, 0.1, 0.1, 0.01, 0.001

def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    return [(voltage - R*current - K_e*omega)/L,
            (K_t*current - b*omega - load_torque)/J]
class PController:
    def __init__(self, kp):
        self.kp = kp
        self.i_term = self.d_term = 0.0
    def update(self, setpoint, measurement, dt):
        self.p_term = self.kp*(setpoint-measurement)
        return self.p_term
def simulate(controller, target=lambda t: 40.0, load=lambda t: 0.0,
             duration=8.0, dt=0.001, noise=0.0, seed=42):
    # Each row describes the beginning of one control interval.
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("duration must be finite and positive")
    if not np.isfinite(noise) or noise < 0:
        raise ValueError("noise must be finite and nonnegative")
    ratio = duration/dt
    if not np.isfinite(ratio):
        raise ValueError("duration/dt must be finite")
    count = round(ratio)
    if count < 1 or not np.isclose(ratio, count, rtol=0, atol=1e-9):
        raise ValueError("duration must be a positive integer multiple of dt")
    rng = np.random.default_rng(seed)
    state = np.array([0.0, 0.0])
    records = []
    for k in range(count):
        t = k*dt
        reference = target(t)
        measured = state[1] + noise*rng.normal()
        voltage = controller.update(reference, measured, dt)
        requested = getattr(controller, "requested", voltage)
        records.append([t, reference, state[0], state[1], measured,
                        requested, voltage, controller.p_term,
                        controller.i_term, controller.d_term])
        result = solve_ivp(motor_dynamics, (t, t+dt), state,
                          args=(voltage, load(t)), rtol=1e-7, atol=1e-9)
        if not result.success:
            raise RuntimeError(result.message)
        state = result.y[:, -1]
    keys = "time target current speed measured requested voltage p i d".split()
    return dict(zip(keys, np.asarray(records).T))

def plot_run(run, title):
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8), layout="constrained")
    axes[0].plot(run['time'], run['target'], '--', label='Target')
    axes[0].plot(run['time'], run['speed'], label='True speed')
    axes[0].set_ylabel('Speed (rad/s)')
    axes[1].plot(run['time'], run['requested'], label='Requested')
    axes[1].plot(run['time'], run['voltage'], '--', label='Applied')
    axes[1].set_ylabel('Voltage (V)')
    for key in ['p', 'i', 'd']:
        axes[2].plot(run['time'], run[key], label=key.upper())
    axes[2].set_ylabel('Contribution (V)')
    axes[2].set_xlabel('Time (s)')
    for ax in axes:
        ax.grid(alpha=.25); ax.legend()
    fig.suptitle(title)
    plt.show()


<img src="assets/lesson-04.svg" alt="Signed area under error samples" style="display:block; width:100%; height:auto; margin:0 auto;">

<div class="figure-caption" style="background-color:rgba(128,128,128,0.10);padding:12px 18px;">

**Integral accumulation (illustrative).** Each sample contributes a signed rectangle. The accumulator holds its existing value when the error is zero. This drawing is a conceptual example, not a motor simulation.

</div>

## 4.1 Accumulate signed error

The integral is signed area under the error curve:

$$I_e(t)=\int_0^t e(s)\,ds,\qquad I_{e,k}=I_{e,k-1}+e_k\Delta t.$$

The discrete expression is a rectangular approximation. Its units are radians when error is rad/s. Positive errors add to the accumulator; negative errors subtract from it. At zero error, the accumulator holds its value. Integral gain has units V/rad.

### Build the accumulator by hand

Consider four successive error samples of 10, 10, −5, and 0 rad/s, each representing 0.1 s. Starting from zero, the contributions are 1, 1, −0.5, and 0 radians. The running totals are:

| Sample | Error (rad/s) | Added area (rad) | Accumulator (rad) |
| --- | --- | --- | --- |
| 1 | 10 | 1 | 1 |
| 2 | 10 | 1 | 2 |
| 3 | −5 | −0.5 | 1.5 |
| 4 | 0 | 0 | 1.5 |

The sign matters: integrating absolute error would keep increasing the correction even after overshoot. Forgetting the time step would make the correction depend on how many samples you happened to take rather than elapsed physical time.

The accumulator is state. It must survive from one controller update to the next, but it must start fresh for an independent experiment. Keep it separate from its voltage contribution so changing integral gain does not obscure what quantity is being stored.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 1 — Accumulate signed error

Accumulate errors [10,10,-5,0] at dt=0.1. Print every accumulator value and check the final value is 1.5.

</div>

<details>
<summary>Show a hint</summary>

Multiply each error by dt before adding; do not sum raw errors.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
accumulator = 0.0
history = []
for error in [10,10,-5,0]:
    accumulator += error*.1
    history.append(accumulator)
np.testing.assert_allclose(history,[1,2,1.5,1.5])
print(history)
```

Zero error stops further accumulation; it does not erase the voltage contribution already earned.

</details>

## 4.2 Combine proportional and integral action

PI control adds the two voltage contributions:

$$V_k=K_p e_k+K_i I_{e,k}.$$

We update the integral before calculating the output. This convention matters when comparing implementations. With $e=10$, $\Delta t=0.1$, $K_p=0.4$, and $K_i=0.5$, the first command is $4+0.5=4.5$ V.

### Trace two PI updates

Start at zero accumulated error. With a first error of 10 rad/s and a 0.1 s interval, the accumulator becomes 1 rad. At gains 0.4 and 0.5, the proportional voltage is 4 V and integral voltage is 0.5 V. Total command is 4.5 V.

If the next measured error is zero, the accumulator stays at 1 rad. The proportional contribution becomes zero while the integral contribution remains 0.5 V. This is precisely the behavior P alone could not provide: a nonzero command with zero present error.

If the next error instead becomes −2 rad/s, the integral decreases by 0.2 rad. It is not instantly reset. This memory makes PI useful and also explains why its transient can overshoot.

Use the error sample at the current update for the rectangular integration approximation. Other integration rules exist, but changing rules while comparing gains would confound the experiment. State the convention once and preserve it across both procedural and class implementations.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 2 — Implement PI control

Implement PIController. Check the first command for the numerical example and verify zero error preserves its integral contribution.

</div>

<details>
<summary>Show a hint</summary>

Store the unscaled integral in the object and expose the scaled contribution as i_term.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
class PIController:
    def __init__(self, kp, ki):
        self.kp, self.ki = kp, ki
        self.integral = 0.0
        self.d_term = 0.0
    def update(self, setpoint, measurement, dt):
        error = setpoint-measurement
        self.integral += error*dt
        self.p_term = self.kp*error
        self.i_term = self.ki*self.integral
        return self.p_term+self.i_term

c = PIController(.4,.5)
assert c.update(10,0,.1)==4.5
assert c.update(0,0,.1)==.5
```

A new controller starts with zero accumulated error; independent experiments need independent controllers.

</details>

## 4.3 Remove the steady residual

If the closed loop settles, the target is achievable, and the integral remains active, a constant nonzero error would keep changing the integral. Thus a true constant equilibrium with integral action requires zero error.

This argument assumes convergence: integral action can destabilize a poorly tuned loop. It also assumes the actuator can deliver the requested voltage.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

Zero error can coexist with nonzero integral output. At 40 rad/s unloaded, the controller must retain approximately 4.4 V to balance friction and back EMF.

</div>

### Why equilibrium requires zero error

If a PI-controlled response truly becomes constant, the voltage and both motor states must stop changing. A constant nonzero error would change the accumulator by the same nonzero amount every interval. That would keep changing voltage, contradicting the assumption of equilibrium. Thus convergence to a constant PI equilibrium requires zero error when integral action is active.

This is not a proof that the system will converge. It might oscillate or diverge; the actuator might saturate; the target might be impossible. The argument tells us what an equilibrium must look like if one is reached under the ideal assumptions.

At 40 rad/s unloaded, torque balance requires 0.4 A and the electrical balance requires 4.4 V. With zero proportional error, that voltage must come from the integral. At $K_i=0.5$, the unscaled integral should approach $4.4/0.5=8.8$ rad.

Plot all three quantities—error, integral contribution, and speed—to connect the mathematical argument to the physical effort.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 3 — Compare P and PI

Compare P gain 0.4 against PI gains 0.4 and 0.5. Check the PI final error is below 0.1 rad/s and smaller than the P error.

</div>

<details>
<summary>Show a hint</summary>

Inspect p, i, and speed together; the integral should supply most of the equilibrium voltage.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
p_run = simulate(PController(.4))
pi_run = simulate(PIController(.4,.5))
assert abs(40-pi_run['speed'][-1]) < .1
assert abs(40-pi_run['speed'][-1]) < abs(40-p_run['speed'][-1])
print('Final integral voltage:',pi_run['i'][-1])
plot_run(pi_run,'PI control: accumulated correction')
```

The proportional contribution shrinks as error vanishes; the integral contribution remains near the required equilibrium voltage.

</details>

## 4.4 Observe the price of integral gain

Stored error persists after the target is reached. If the integral has grown too large during acceleration, it continues driving the motor and can cause overshoot. Negative error must then unwind that accumulation. Increasing $K_i$ can shorten error correction while worsening overshoot or oscillation.

### Read overshoot as stored effort

During startup, speed remains below target for a while, so the integral grows. By the time speed reaches the target, the integral may command more than the equilibrium voltage. The motor continues accelerating until negative error reduces the stored correction enough.

A smaller integral gain can take longer to erase the residual left by P. In the supplied sweep, $K_i=0.2$ still has a noticeable finite-time error at eight seconds. That is not evidence that integral action necessarily leaves a permanent residual; inspect whether the response is still moving and extend the observation if necessary.

A larger gain reduces one kind of delay but can enlarge overshoot. Compare peaks, time spent away from target, and integral voltage rather than selecting whichever curve first crosses the target.

The supplied setup includes the sampled simulator from Lesson 3. Its logging is unchanged: the `i` field means integral voltage contribution, while `current` means electrical winding current. Their similar names do not make them the same physical quantity.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Compare integral gains

Compare Ki=0.2, 0.5, and 1.0 with Kp=0.4. Overlay speed and integral output. Report peak speed and final error without assuming larger gain is better.

</div>

<details>
<summary>Show a hint</summary>

Use two shared-time panels and a fresh PIController each time.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
fig, axes = plt.subplots(2,1,sharex=True,figsize=(10,6),layout='constrained')
for ki in [.2,.5,1.0]:
    trial=simulate(PIController(.4,ki))
    axes[0].plot(trial['time'],trial['speed'],label=f'Ki={ki}')
    axes[1].plot(trial['time'],trial['i'],label=f'Ki={ki}')
    print(ki, trial['speed'].max(),40-trial['speed'][-1])
axes[0].axhline(40,ls='--',color='gray');axes[0].set_ylabel('Speed (rad/s)')
axes[1].set_ylabel('Integral voltage (V)');axes[1].set_xlabel('Time (s)')
for ax in axes:ax.legend();ax.grid(alpha=.25)
plt.show()
```

The plotted accumulator explains overshoot more directly than the speed curve alone. Saturation and windup will be treated explicitly in Lesson 9.

</details>

## What the experiment establishes

Integral action supplies persistent correction without requiring persistent error. Your gain sweep also shows why correcting residual error and producing a well-damped transient are separate objectives. Next, we will add sensitivity to how quickly error is changing.

## Before moving on

Run your exercises from a fresh kernel in order. Check the stated numerical conditions and explain your plots in your own words: what changed, why it changed, and which model assumptions limit the conclusion. Reference solutions are one implementation, not the only correct approach.

[Previous lesson](03_proportional_control.ipynb) · [Next lesson](05_derivative_control.ipynb)

## References and further reading

[Åström and Murray, Feedback Systems, Chapter 11](https://www.cds.caltech.edu/~murray/books/AM08/pdf/fbs-pid_24Jul2020.pdf) — additional background on PID structure and implementation. Our examples use independent parallel gains and the conventions stated in this lesson.